In [1]:
"""
COMPREHENSIVE HYPOTHESIS TESTING GUIDE (Python / SciPy / Statsmodels)
======================================================================
Covers: assumptions checks, parametric tests, non-parametric tests,
proportion tests, correlation tests, ANOVA, post-hoc tests, effect sizes,
power analysis, and multiple-comparison correction.
 
Install once:
    pip install scipy numpy pandas --break-system-packages
 
Pure SciPy/NumPy implementation — no statsmodels dependency required.
"""
 
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
 
np.random.seed(42)

In [6]:
# ======================================================================
# 0. THE FRAMEWORK — what hypothesis testing actually does
# ======================================================================
print(f"""
Every test follows the same logic:

1. State H0 (null: "no effect/no difference") and H1 (alternative).
2. Choose significance level alpha (usually 0.05).
3. Check the test's ASSUMPTIONS (normality, variance, independence...).
4. Compute a test statistic and its p-value.
5. p-value = P(seeing data this extreme | H0 is true).
   - p < alpha  -> reject H0 ("statistically significant")
   - p >= alpha -> fail to reject H0 (NOT "H0 is true")
6. Report an EFFECT SIZE alongside p-value — significance != importance.

Picking the right test depends on:

- Data type (continuous, ordinal, categorical/binary)
- Number of groups (1, 2, 3+)
- Independent or paired/repeated samples
- Whether normality/equal-variance assumptions hold
""")


Every test follows the same logic:

1. State H0 (null: "no effect/no difference") and H1 (alternative).
2. Choose significance level alpha (usually 0.05).
3. Check the test's ASSUMPTIONS (normality, variance, independence...).
4. Compute a test statistic and its p-value.
5. p-value = P(seeing data this extreme | H0 is true).
   - p < alpha  -> reject H0 ("statistically significant")
   - p >= alpha -> fail to reject H0 (NOT "H0 is true")
6. Report an EFFECT SIZE alongside p-value — significance != importance.

Picking the right test depends on:

- Data type (continuous, ordinal, categorical/binary)
- Number of groups (1, 2, 3+)
- Independent or paired/repeated samples
- Whether normality/equal-variance assumptions hold



In [7]:

# ======================================================================
# 1. ASSUMPTION CHECKS (run these BEFORE choosing a parametric test)
# ======================================================================
 
def check_normality(data, name="sample"):
    """Shapiro-Wilk: H0 = data is normally distributed.
    Best for n < 50; for larger n use D'Agostino-Pearson."""
    stat, p = stats.shapiro(data)
    print(f"[Shapiro-Wilk] {name}: stat={stat:.4f}, p={p:.4f} -> "
          f"{'normal' if p > 0.05 else 'NOT normal'} (alpha=0.05)")
    return p > 0.05
 
def check_equal_variance(a, b, name="groups"):
    """Levene's test: H0 = variances are equal. More robust to
    non-normality than Bartlett's test."""
    stat, p = stats.levene(a, b)
    print(f"[Levene] {name}: stat={stat:.4f}, p={p:.4f} -> "
          f"{'equal variance' if p > 0.05 else 'UNEQUAL variance'}")
    return p > 0.05
 
# Example data
group_a = np.random.normal(50, 10, 30)
group_b = np.random.normal(55, 12, 30)
 
check_normality(group_a, "group_a")
check_normality(group_b, "group_b")
check_equal_variance(group_a, group_b)

[Shapiro-Wilk] group_a: stat=0.9751, p=0.6868 -> normal (alpha=0.05)
[Shapiro-Wilk] group_b: stat=0.9837, p=0.9130 -> normal (alpha=0.05)
[Levene] groups: stat=2.0687, p=0.1557 -> equal variance


np.True_

In [8]:
# ======================================================================
# 2. ONE-SAMPLE TESTS — compare a sample to a known/hypothesized value
# ======================================================================
 
def one_sample_t_test(data, popmean, alpha=0.05):
    """Use when: continuous data, sample roughly normal, population
    std unknown (almost always the case)."""
    t_stat, p = stats.ttest_1samp(data, popmean)
    d = (np.mean(data) - popmean) / np.std(data, ddof=1)  # Cohen's d
    print(f"One-sample t-test: t={t_stat:.4f}, p={p:.4f}, Cohen's d={d:.3f} "
          f"-> {'reject H0' if p < alpha else 'fail to reject H0'}")
    return t_stat, p
 
one_sample_t_test(group_a, popmean=48)
 
def one_sample_z_test(data, popmean, pop_std):
    """Use when population std IS known (rare) or n is large (>30)
    and you want to use the normal distribution instead of t."""
    n = len(data)
    z = (np.mean(data) - popmean) / (pop_std / np.sqrt(n))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f"One-sample z-test: z={z:.4f}, p={p:.4f}")
    return z, p

One-sample t-test: t=0.0721, p=0.9430, Cohen's d=0.013 -> fail to reject H0


In [9]:

# ======================================================================
# 3. TWO-SAMPLE TESTS — compare two groups
# ======================================================================
 
def independent_t_test(a, b, equal_var=True, alpha=0.05):
    """Independent (unpaired) samples, e.g. control vs treatment group,
    two different customer segments. equal_var=False -> Welch's t-test
    (safer default when variances differ or sample sizes differ)."""
    t_stat, p = stats.ttest_ind(a, b, equal_var=equal_var)
    pooled_std = np.sqrt(((len(a)-1)*np.var(a, ddof=1) +
                           (len(b)-1)*np.var(b, ddof=1)) / (len(a)+len(b)-2))
    d = (np.mean(a) - np.mean(b)) / pooled_std
    label = "Student's t" if equal_var else "Welch's t"
    print(f"{label}-test: t={t_stat:.4f}, p={p:.4f}, Cohen's d={d:.3f} "
          f"-> {'reject H0' if p < alpha else 'fail to reject H0'}")
    return t_stat, p
 
independent_t_test(group_a, group_b, equal_var=True)   # standard
independent_t_test(group_a, group_b, equal_var=False)  # Welch's (recommended default)
 
def paired_t_test(before, after, alpha=0.05):
    """Same subjects measured twice, e.g. before/after an intervention,
    churn score pre/post a retention campaign."""
    t_stat, p = stats.ttest_rel(before, after)
    diff = np.array(after) - np.array(before)
    d = np.mean(diff) / np.std(diff, ddof=1)
    print(f"Paired t-test: t={t_stat:.4f}, p={p:.4f}, Cohen's d={d:.3f}")
    return t_stat, p
 
before = np.random.normal(60, 8, 25)
after = before + np.random.normal(3, 5, 25)  # simulate improvement
paired_t_test(before, after)

Student's t-test: t=-2.0720, p=0.0427, Cohen's d=-0.535 -> reject H0
Welch's t-test: t=-2.0720, p=0.0429, Cohen's d=-0.535 -> reject H0
Paired t-test: t=-3.8041, p=0.0009, Cohen's d=0.761


(np.float64(-3.804068700058591), np.float64(0.0008631807921069468))

In [10]:
# ======================================================================
# 4. NON-PARAMETRIC ALTERNATIVES (use when normality fails, or ordinal
#    data, or small/skewed samples)
# ======================================================================
 
def mann_whitney_u(a, b, alpha=0.05):
    """Non-parametric alternative to independent t-test. Compares
    distributions/medians via rank sums, no normality assumption."""
    stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    print(f"Mann-Whitney U: U={stat:.4f}, p={p:.4f}")
    return stat, p
 
def wilcoxon_signed_rank(before, after, alpha=0.05):
    """Non-parametric alternative to paired t-test."""
    stat, p = stats.wilcoxon(before, after)
    print(f"Wilcoxon signed-rank: W={stat:.4f}, p={p:.4f}")
    return stat, p
 
def kruskal_wallis(*groups, alpha=0.05):
    """Non-parametric alternative to one-way ANOVA (3+ groups)."""
    stat, p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H: H={stat:.4f}, p={p:.4f}")
    return stat, p
 
mann_whitney_u(group_a, group_b)
wilcoxon_signed_rank(before, after)

Mann-Whitney U: U=310.0000, p=0.0392
Wilcoxon signed-rank: W=41.0000, p=0.0006


(np.float64(41.0), np.float64(0.0005564093589782715))

In [11]:

# ======================================================================
# 5. CATEGORICAL DATA — chi-square and proportion tests
# ======================================================================
 
def chi_square_independence(contingency_table, alpha=0.05):
    """H0: the two categorical variables are independent.
    e.g. does 'plan type' relate to 'churned (yes/no)'?"""
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    print(f"Chi-square test of independence: chi2={chi2:.4f}, dof={dof}, p={p:.4f}")
    print("Expected frequencies:\n", expected)
    if (expected < 5).any():
        print("WARNING: some expected counts < 5 — consider Fisher's exact test")
    return chi2, p
 
# Example: churn (rows) vs contract type (cols)
table = pd.DataFrame(
    [[120, 40, 10], [30, 60, 90]],
    index=["Churned", "Retained"],
    columns=["Month-to-month", "One-year", "Two-year"]
)
chi_square_independence(table.values)
 
def chi_square_goodness_of_fit(observed, expected, alpha=0.05):
    """H0: observed frequencies match an expected distribution."""
    chi2, p = stats.chisquare(observed, f_exp=expected)
    print(f"Chi-square GOF: chi2={chi2:.4f}, p={p:.4f}")
    return chi2, p
 
def fisher_exact_test(table_2x2, alpha=0.05):
    """Use for 2x2 tables with small expected counts (<5)."""
    odds_ratio, p = stats.fisher_exact(table_2x2)
    print(f"Fisher's exact: OR={odds_ratio:.4f}, p={p:.4f}")
    return odds_ratio, p
 
def two_proportion_z_test(count1, nobs1, count2, nobs2, alpha=0.05):
    """Compare two proportions/rates, e.g. conversion rate A vs B.
    Pooled-proportion z-test (standard A/B test formula)."""
    p1, p2 = count1 / nobs1, count2 / nobs2
    p_pool = (count1 + count2) / (nobs1 + nobs2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/nobs1 + 1/nobs2))
    z = (p1 - p2) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f"Two-proportion z-test: p1={p1:.3f}, p2={p2:.3f}, z={z:.4f}, p={p:.4f}")
    return z, p
 
two_proportion_z_test(count1=45, nobs1=500, count2=60, nobs2=520)  # e.g. A/B test conversions
 
def mcnemar_test(table_2x2, alpha=0.05):
    """Paired categorical data, e.g. same customers' yes/no before vs
    after a UI change (McNemar, not chi-square, because paired).
    table_2x2 = [[both_yes, yes_to_no], [no_to_yes, both_no]]
    Only the off-diagonal discordant pairs (b, c) matter."""
    b, c = table_2x2[0][1], table_2x2[1][0]
    if b + c < 25:
        p = min(2 * stats.binom.cdf(min(b, c), b + c, 0.5), 1.0)  # exact binomial
        print(f"McNemar's test (exact): b={b}, c={c}, p={p:.4f}")
        return None, p
    stat = (abs(b - c) - 1)**2 / (b + c)  # chi2 approx with continuity correction
    p = 1 - stats.chi2.cdf(stat, df=1)
    print(f"McNemar's test (chi2 approx): stat={stat:.4f}, p={p:.4f}")
    return stat, p

Chi-square test of independence: chi2=121.8137, dof=2, p=0.0000
Expected frequencies:
 [[72.85714286 48.57142857 48.57142857]
 [77.14285714 51.42857143 51.42857143]]
Two-proportion z-test: p1=0.090, p2=0.115, z=-1.3337, p=0.1823


In [12]:

# ======================================================================
# 6. ANOVA — comparing 3+ group means
# ======================================================================
 
def one_way_anova(*groups, alpha=0.05):
    """H0: all group means are equal. Assumes normality + equal
    variance across groups (check with Levene's test first)."""
    f_stat, p = stats.f_oneway(*groups)
    # eta-squared effect size
    all_data = np.concatenate(groups)
    grand_mean = np.mean(all_data)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((all_data - grand_mean)**2)
    eta_sq = ss_between / ss_total
    print(f"One-way ANOVA: F={f_stat:.4f}, p={p:.4f}, eta^2={eta_sq:.3f}")
    return f_stat, p
 
group_c = np.random.normal(52, 9, 30)
one_way_anova(group_a, group_b, group_c)
 
def posthoc_pairwise(groups_dict, alpha=0.05):
    """Run AFTER a significant ANOVA to find WHICH pairs differ.
    Pairwise independent t-tests with Bonferroni correction
    (a practical stand-in for Tukey HSD; Tukey is slightly less
    conservative but needs statsmodels)."""
    names = list(groups_dict.keys())
    pairs = list(combinations(names, 2))
    raw_p = []
    for n1, n2 in pairs:
        _, p = stats.ttest_ind(groups_dict[n1], groups_dict[n2])
        raw_p.append(p)
    corrected = np.minimum(np.array(raw_p) * len(pairs), 1.0)  # Bonferroni
    for (n1, n2), p_raw, p_corr in zip(pairs, raw_p, corrected):
        flag = "significant" if p_corr < alpha else "n.s."
        print(f"{n1} vs {n2}: raw p={p_raw:.4f}, Bonferroni p={p_corr:.4f} -> {flag}")
    return dict(zip(pairs, corrected))
 
posthoc_pairwise({"A": group_a, "B": group_b, "C": group_c})
 
def two_way_anova_note():
    """Two-way ANOVA (two categorical factors + interaction) needs a
    full linear-model decomposition that's impractical to hand-roll
    reliably. If you need it: pip install statsmodels, then:
 
        from statsmodels.formula.api import ols
        import statsmodels.api as sm
        model = ols('dv ~ C(factor1) * C(factor2)', data=df).fit()
        print(sm.stats.anova_lm(model, typ=2))
    """
    print("Two-way ANOVA requires statsmodels — see function docstring.")
 
two_way_anova_note()

One-way ANOVA: F=2.2617, p=0.1103, eta^2=0.049
A vs B: raw p=0.0427, Bonferroni p=0.1282 -> n.s.
A vs C: raw p=0.1442, Bonferroni p=0.4326 -> n.s.
B vs C: raw p=0.5089, Bonferroni p=1.0000 -> n.s.
Two-way ANOVA requires statsmodels — see function docstring.


In [13]:

# ======================================================================
# 7. CORRELATION TESTS
# ======================================================================
 
def pearson_correlation(x, y, alpha=0.05):
    """Linear relationship, both variables continuous & normal."""
    r, p = stats.pearsonr(x, y)
    print(f"Pearson r={r:.4f}, p={p:.4f}")
    return r, p
 
def spearman_correlation(x, y, alpha=0.05):
    """Monotonic relationship, ordinal data or non-normal, robust to
    outliers (rank-based)."""
    rho, p = stats.spearmanr(x, y)
    print(f"Spearman rho={rho:.4f}, p={p:.4f}")
    return rho, p
 
x = np.random.normal(0, 1, 50)
y = 2*x + np.random.normal(0, 1, 50)
pearson_correlation(x, y)
spearman_correlation(x, y)

Pearson r=0.8698, p=0.0000
Spearman rho=0.8648, p=0.0000


(np.float64(0.8647779111644658), np.float64(5.747304523217858e-16))